<a href="https://colab.research.google.com/github/dibyanshulunia25/Lung-Tumor/blob/main/Lung_Tumor_FL.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Revised Code with increased number of clients and epochs as well as added some more features

### Block 1: Setup & Environment Initialization

In [3]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader, Subset
from sklearn.metrics import precision_score, recall_score, f1_score, roc_auc_score, classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import copy
import os
import random
from PIL import Image
from google.colab import drive

Block 2: Advanced Preprocessing & Augmentation
### We use specific transforms to make the model robust against different scanner types and slide orientations.

In [4]:
def get_transforms(train=False):
    # Data Augmentation is crucial for Histopathology.
    # Rotation and Flips simulate how slides can be placed in a scanner.

    if train:
        return transforms.Compose([
            transforms.Resize((224, 224)),
            transforms.RandomHorizontalFlip(),
            transforms.RandomVerticalFlip(),
            transforms.RandomRotation(90), # 90-degree rotations for medical slides
            transforms.ToTensor(),
            # Normalizing using ImageNet means and standard deviations
            transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
        ])
    else:
        return transforms.Compose([
            transforms.Resize((224, 224)),
            transforms.ToTensor(),
            transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
        ])

Block 3: Realistic Non-IID Data Partitioning
### This block creates the 60/10/30 split and ensures each hospital has a unique "biases" (Non-IID distribution).

In [10]:
def prepare_federated_data(num_clients=5, alpha=0.5):

    train_transform = get_transforms(train=True)
    val_test_transform = get_transforms(train=False)

    data_dir = '/content/drive/MyDrive/Mini Project Dataset/Lung (Adenocarcinoma)'

    base_dataset = datasets.ImageFolder(data_dir)
    class_names = base_dataset.classes
    labels = np.array(base_dataset.targets)

    # ------------------------------------------------
    # GLOBAL SPLIT (70 TRAIN / 15 VAL / 15 TEST)
    # ------------------------------------------------

    class_indices = [np.where(labels == i)[0] for i in range(len(class_names))]

    train_pool_idx = []
    val_idx = []
    test_idx = []

    for c in range(len(class_names)):

        idx = class_indices[c]
        np.random.shuffle(idx)

        total = len(idx)

        test_split = int(total * 0.15)
        val_split = int(total * 0.15)

        test_idx.extend(idx[:test_split])
        val_idx.extend(idx[test_split:test_split + val_split])
        train_pool_idx.extend(idx[test_split + val_split:])

    train_pool_idx = np.array(train_pool_idx)

    # ------------------------------------------------
    # GLOBAL TEST LOADER
    # ------------------------------------------------

    test_ds = copy.deepcopy(base_dataset)
    test_ds.transform = val_test_transform

    test_loader = DataLoader(
        Subset(test_ds, test_idx),
        batch_size=128
    )

    # ------------------------------------------------
    # GLOBAL VALIDATION LOADER
    # ------------------------------------------------

    val_ds = copy.deepcopy(base_dataset)
    val_ds.transform = val_test_transform

    val_loader = DataLoader(
        Subset(val_ds, val_idx),
        batch_size=128
    )

    # ------------------------------------------------
    # DIRICHLET NON-IID CLIENT SPLIT
    # ------------------------------------------------

    def dirichlet_split(labels, alpha, num_clients):

        labels = np.array(labels)
        n_classes = labels.max() + 1

        class_indices = [np.where(labels == y)[0] for y in range(n_classes)]

        client_indices = [[] for _ in range(num_clients)]

        for c in range(n_classes):

            np.random.shuffle(class_indices[c])

            proportions = np.random.dirichlet(
                np.repeat(alpha, num_clients)
            )

            proportions = (np.cumsum(proportions) * len(class_indices[c])).astype(int)[:-1]

            split = np.split(class_indices[c], proportions)

            for client_id, idx in enumerate(split):
                client_indices[client_id].extend(idx)

        return client_indices


    client_indices = dirichlet_split(
        labels=[labels[i] for i in train_pool_idx],
        alpha=alpha,
        num_clients=num_clients
    )

    # ------------------------------------------------
    # CREATE CLIENT LOADERS
    # ------------------------------------------------

    client_loaders = []

    print("\n" + "="*70)
    print(f"{'FEDERATED DATA DISTRIBUTION REPORT':^70}")
    print("="*70)

    print(f"Total Dataset Size : {len(base_dataset)}")
    print(f"Training Pool (70%) : {len(train_pool_idx)}")
    print(f"Validation Set (15%): {len(val_idx)}")
    print(f"Test Set (15%)      : {len(test_idx)}")

    print("-"*70)

    for i in range(num_clients):

        c_idx = train_pool_idx[client_indices[i]]

        train_ds = copy.deepcopy(base_dataset)
        train_ds.transform = train_transform

        client_loaders.append({
            'train': DataLoader(
                Subset(train_ds, c_idx),
                batch_size=16,
                shuffle=True
            )
        })

        # Count class distribution
        class_count = {cls:0 for cls in class_names}

        for idx in c_idx:
            label = labels[idx]
            class_count[class_names[label]] += 1

        print(f"\nClient {i+1}")
        print(f"Training Samples : {len(c_idx)}")

        print("Class Distribution:")
        for cls in class_names:
            print(f"   {cls:<15}: {class_count[cls]}")

    print("\n" + "="*70 + "\n")

    return client_loaders, val_loader, test_loader, class_names



In [11]:
clients, val_loader, test_loader, class_names = prepare_federated_data(
    num_clients=5,
    alpha=0.5
)


                  FEDERATED DATA DISTRIBUTION REPORT                  
Total Dataset Size : 15001
Training Pool (70%) : 10501
Validation Set (15%): 2250
Test Set (15%)      : 2250
----------------------------------------------------------------------

Client 1
Training Samples : 4316
Class Distribution:
   adenocarcinoma : 2545
   benign         : 50
   squamous_cell_carcinoma: 1721

Client 2
Training Samples : 311
Class Distribution:
   adenocarcinoma : 133
   benign         : 85
   squamous_cell_carcinoma: 93

Client 3
Training Samples : 2450
Class Distribution:
   adenocarcinoma : 21
   benign         : 772
   squamous_cell_carcinoma: 1657

Client 4
Training Samples : 307
Class Distribution:
   adenocarcinoma : 284
   benign         : 5
   squamous_cell_carcinoma: 18

Client 5
Training Samples : 3117
Class Distribution:
   adenocarcinoma : 518
   benign         : 2588
   squamous_cell_carcinoma: 11




Block 4: Architecture & Evaluation Metrics
### Using MobileNetV2 with Transfer Learning and unfreezing deeper layers.

In [12]:
def get_efficient_model():
    """Returns a fine-tuned MobileNetV2 for Lung Cancer classification."""
    model = models.mobilenet_v2(weights=models.MobileNetV2_Weights.DEFAULT)

    # Initial freeze of all layers
    for param in model.parameters():
        param.requires_grad = False

    # Unfreeze the last 2 feature extraction blocks to adapt to medical textures
    for param in model.features[-2:].parameters():
        param.requires_grad = True

    # Redefine classifier for 3 target classes
    model.classifier[1] = nn.Linear(model.last_channel, 3)
    return model

def evaluate(model, loader):
    """Calculates Loss, Accuracy, and AUC-ROC on provided loader."""
    model.eval()
    y_true, y_pred, y_probs = [], [], []
    total_loss, criterion = 0, nn.CrossEntropyLoss()

    with torch.no_grad():
        for imgs, labels in loader:
            outputs = model(imgs)
            total_loss += criterion(outputs, labels).item()
            probs = torch.softmax(outputs, dim=1)
            preds = torch.argmax(outputs, dim=1)
            y_true.extend(labels.cpu().numpy())
            y_pred.extend(preds.cpu().numpy())
            y_probs.extend(probs.cpu().numpy())

    acc = np.mean(np.array(y_pred) == np.array(y_true)) * 100
    auc = roc_auc_score(y_true, y_probs, multi_class='ovr')
    return total_loss/len(loader), acc, auc

Block 5: Federated Orchestrator with Stochastic Participation
### This is the core training loop with Early Stopping, Client Dropout, and Nested Epoch Logging.

In [14]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader, Subset
from sklearn.metrics import precision_score, recall_score, f1_score, roc_auc_score, classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import copy
import os
import random
from PIL import Image
from google.colab import drive

def get_efficient_model():
    """Returns a fine-tuned MobileNetV2 for Lung Cancer classification."""
    try:
        model = models.mobilenet_v2(weights=models.MobileNetV2_Weights.DEFAULT)
    except AttributeError:
        # Fallback for older torchvision versions
        print("Warning: MobileNetV2_Weights.DEFAULT not found. Falling back to pretrained=True.")
        model = models.mobilenet_v2(pretrained=True)

    # Initial freeze of all layers
    for param in model.parameters():
        param.requires_grad = False

    # Unfreeze the last 2 feature extraction blocks to adapt to medical textures
    for param in model.features[-2:].parameters():
        param.requires_grad = True

    # Redefine classifier for 3 target classes
    model.classifier[1] = nn.Linear(model.last_channel, 3)
    return model

def evaluate(model, loader):
    """Calculates Loss, Accuracy, and AUC-ROC on provided loader."""
    model.eval()
    y_true, y_pred, y_probs = [], [], []
    total_loss, criterion = 0, nn.CrossEntropyLoss()

    # Ensure model is on the correct device for evaluation
    current_device = next(model.parameters()).device # Get the device where model parameters are

    with torch.no_grad():
        for imgs, labels in loader:
            # Move images and labels to the same device as the model
            imgs, labels = imgs.to(current_device), labels.to(current_device)
            outputs = model(imgs)
            total_loss += criterion(outputs, labels).item()
            probs = torch.softmax(outputs, dim=1)
            preds = torch.argmax(outputs, dim=1)
            y_true.extend(labels.cpu().numpy())
            y_pred.extend(preds.cpu().numpy())
            y_probs.extend(probs.cpu().numpy())

    acc = np.mean(np.array(y_pred) == np.array(y_true)) * 100
    auc = roc_auc_score(y_true, y_probs, multi_class='ovr')
    return total_loss/len(loader), acc, auc

# --- Training Configuration ---
local_epochs = 5
max_rounds = 40
patience = 2
num_clients = len(clients)
counter, best_acc = 0, 0.0
history = {'acc': [], 'loss': [], 'auc': []}
dropout_limit = 3
dropout_rounds_in_window = 0

# Set device for GPU acceleration
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
global_model = get_efficient_model().to(device)

print(f"Starting High-Speed Federated Training on {device}...")

for r in range(max_rounds):
    if r % 10 == 0: dropout_rounds_in_window = 0

    print(f"\n{'#'*65}\n# ROUND {r+1}/{max_rounds}\n{'#'*65}")

    # --- Dropout Logic: EXACTLY 1 client misses in dropout rounds ---
    active_indices = list(range(num_clients))
    if dropout_rounds_in_window < dropout_limit and random.random() < 0.3:
        dropped = random.randint(0, num_clients - 1)
        active_indices.remove(dropped)
        dropout_rounds_in_window += 1
        print(f"[NETWORK ALERT] Hospital {dropped + 1} OFFLINE. Proceeding with {len(active_indices)} nodes.")
    else:
        print(f"[STATUS] Network Stable. All {num_clients} hospitals participating.")

    local_weights = []

    # --- 1. LOCAL TRAINING STAGE ---
    for i in active_indices:
        print(f"\n[Hospital {i+1} Local Training]")
        # Fast copy and move to GPU
        local_model = copy.deepcopy(global_model).to(device)
        optimizer = optim.Adam(filter(lambda p: p.requires_grad, local_model.parameters()), lr=1e-4, weight_decay=1e-5)
        criterion = nn.CrossEntropyLoss()

        local_model.train()
        for epoch in range(local_epochs):
            r_loss, correct, total = 0.0, 0, 0
            for imgs, labels in clients[i]['train']:
                imgs, labels = imgs.to(device), labels.to(device) # Move data to GPU
                optimizer.zero_grad()
                out = local_model(imgs)
                loss = criterion(out, labels)
                loss.backward()
                optimizer.step()

                r_loss += loss.item()
                total += labels.size(0)
                correct += torch.argmax(out, 1).eq(labels).sum().item()

            print(f"   Epoch {epoch+1}/{local_epochs} | Loss: {r_loss/len(clients[i]['train']):.4f} | Acc: {100.*correct/total:.2f}%")

        # Move weights to CPU immediately to free GPU VRAM
        local_weights.append({k: v.cpu() for k, v in local_model.state_dict().items()})

        # Explicit Memory Cleanup
        del local_model
        torch.cuda.empty_cache()

    # --- 2. VECTORIZED AGGREGATION (FASTEST METHOD) ---
    print("\n--- SERVER: Performing Vectorized FedAvg (Parallel Processing) ---")
    with torch.no_grad():
        new_global_dict = {}
        for key in global_model.state_dict().keys():
            client_tensors = [client[key] for client in local_weights]
            # Check if the tensor is a floating point type before averaging
            if client_tensors[0].dtype in [torch.float32, torch.float64, torch.float16, torch.bfloat16]:
                new_global_dict[key] = torch.stack(client_tensors, dim=0).mean(dim=0)
            else:
                # For non-floating point types (like num_batches_tracked), just take the value from the first client
                new_global_dict[key] = client_tensors[0]

        # Update the global model with averaged weights
        global_model.load_state_dict(new_global_dict)

    # --- 3. GLOBAL EVALUATION ---
    g_loss, g_acc, g_auc = evaluate(global_model, test_loader)
    history['acc'].append(g_acc)
    history['loss'].append(g_loss)
    history['auc'].append(g_auc)

    print(f"\n>>> ROUND {r+1} GLOBAL RESULTS: Acc: {g_acc:.2f}% | AUC: {g_auc:.3f}")

    # Early Stopping check
    if g_acc > best_acc:
        best_acc, counter = g_acc, 0
        best_model_wts = copy.deepcopy(global_model.state_dict())
        print(f"*** New Peak Accuracy! Saving best model. ***")
    else:
        counter += 1
        print(f"   (Patience Tracker: {counter}/{patience})")

    if counter >= patience:
        print(f"\n[EARLY STOP] No improvement for {patience} rounds. Reverting to best model.")
        global_model.load_state_dict(best_model_wts)
        break

Starting High-Speed Federated Training on cuda...

#################################################################
# ROUND 1/40
#################################################################
[NETWORK ALERT] Hospital 1 OFFLINE. Proceeding with 4 nodes.

[Hospital 2 Local Training]


/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=MobileNet_V2_Weights.IMAGENET1K_V1`. You can also use `weights=MobileNet_V2_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


   Epoch 1/5 | Loss: 0.6580 | Acc: 77.17%
   Epoch 2/5 | Loss: 0.2832 | Acc: 91.32%
   Epoch 3/5 | Loss: 0.2209 | Acc: 94.86%
   Epoch 4/5 | Loss: 0.2212 | Acc: 91.32%
   Epoch 5/5 | Loss: 0.1647 | Acc: 95.18%

[Hospital 3 Local Training]
   Epoch 1/5 | Loss: 0.1120 | Acc: 97.39%
   Epoch 2/5 | Loss: 0.0472 | Acc: 98.90%
   Epoch 3/5 | Loss: 0.0242 | Acc: 99.18%
   Epoch 4/5 | Loss: 0.0316 | Acc: 99.02%
   Epoch 5/5 | Loss: 0.0589 | Acc: 99.10%

[Hospital 4 Local Training]
   Epoch 1/5 | Loss: 0.5838 | Acc: 85.34%
   Epoch 2/5 | Loss: 0.2592 | Acc: 92.51%
   Epoch 3/5 | Loss: 0.1688 | Acc: 93.81%
   Epoch 4/5 | Loss: 0.1470 | Acc: 96.09%
   Epoch 5/5 | Loss: 0.1244 | Acc: 97.07%

[Hospital 5 Local Training]
   Epoch 1/5 | Loss: 0.1294 | Acc: 96.37%
   Epoch 2/5 | Loss: 0.0321 | Acc: 99.29%
   Epoch 3/5 | Loss: 0.0221 | Acc: 99.29%
   Epoch 4/5 | Loss: 0.0198 | Acc: 99.42%
   Epoch 5/5 | Loss: 0.0155 | Acc: 99.49%

--- SERVER: Performing Vectorized FedAvg (Parallel Processing) ---

>>> 

Block 6: Final Performance Analysis & Visualization
### Run this to generate your project documentation graphs and the confusion matrix.

In [ ]:
# 1. Visualization of Learning Progress
plt.figure(figsize=(15, 5))
plt.subplot(1, 3, 1); plt.plot(history['acc'], color='forestgreen', marker='o'); plt.title('Test Accuracy'); plt.xlabel('Round')
plt.subplot(1, 3, 2); plt.plot(history['loss'], color='darkred', marker='o'); plt.title('Test Loss'); plt.xlabel('Round')
plt.subplot(1, 3, 3); plt.plot(history['auc'], color='navy', marker='o'); plt.title('AUC-ROC Curve'); plt.xlabel('Round')
plt.tight_layout(); plt.show()

# 2. Final Evaluation Report
y_true, y_pred = [], []
global_model.eval()
with torch.no_grad():
    for imgs, labels in global_test_loader:
        y_true.extend(labels.cpu().numpy())
        y_pred.extend(torch.argmax(global_model(imgs), dim=1).cpu().numpy())

print("\n" + "="*50 + "\nFINAL GLOBAL MODEL PERFORMANCE REPORT\n" + "="*50)
print(classification_report(y_true, y_pred, target_names=class_names))

# 3. Final Confusion Matrix
cm = confusion_matrix(y_true, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Greens', xticklabels=class_names, yticklabels=class_names)
plt.title('Final Confusion Matrix (30% Unseen Data)')
plt.ylabel('Ground Truth')
plt.xlabel('Prediction')
plt.show()